## More automatic TSM generation

#### Imports

In [ ]:
import sys, os
basedir = ''
if "__file__" in globals(): basedir = os.path.dirname(__file__)
sys.path.insert(0, os.path.join(basedir, os.path.pardir, os.path.pardir, 'python'))

In [ ]:
import pandas as pd
import numpy as np
import scipy as sci
import matplotlib.pyplot as pl
import matplotlib.image as img
import subprocess
import pathlib
import pyvista as pv
import copy

import fenics_sz.utils
output_folder = pathlib.Path(os.path.join(basedir, "output"))
output_folder.mkdir(exist_ok=True, parents=True)

In [ ]:
from fenics_sz.sz_problems.sz_slab import create_slab, plot_slab
from fenics_sz.sz_problems.sz_geometry import create_sz_geometry
from fenics_sz.sz_problems.sz_steady_dislcreep import SteadyDislSubductionProblem
from fenics_sz.sz_problems.sz_tdep_dislcreep import TDDislSubductionProblem
from fenics_sz.sz_problems.sz_params import default_params, allsz_params

In [ ]:
from fenics_sz.fluid_release.perple_x_integration import get_PT_data_from_tabs, plot_PT_data
import fenics_sz.fluid_release.get_PT_curves 

In [ ]:
from fenics_sz.fluid_release.workflow_functions import (in_domain, get_st_grid, get_interpolator, predict_h2o,
                                                          Cell, remove_rehydration, get_water_loss, sorted_water_loss_by_layer,
                                                          get_TSMstye_line)

#### Read Perple_X data

In [ ]:
DMM_data = get_PT_data_from_tabs('DMMdamp_25')
uvolcs_data = get_PT_data_from_tabs('upvolc_25')
lvolcs_data = get_PT_data_from_tabs('lovolc_25')
dikes_data = get_PT_data_from_tabs('dike_25')
gabbros_data = get_PT_data_from_tabs('gabbro_25')

In [ ]:
interps = []
interps.append(get_interpolator(uvolcs_data))
interps.append(get_interpolator(lvolcs_data))
interps.append(get_interpolator(dikes_data))
interps.append(get_interpolator(gabbros_data))
interps.append(get_interpolator(DMM_data))

## Full TSM

In [ ]:
resscale = 5.0
u_res = 100
h_serp = 2.0

In [ ]:
global_water_losses = []
global_water_losses_and_depths = []
layer_water_losses = []
global_flux_in = []
global_flux_out =[]
global_flux_out_sanity_check = []

for key in allsz_params.keys():
    sz_dict = allsz_params[key]
    water_loss, water_losses_and_depths, layer_losses, layer_losses_and_depths, flux_in, flux_out, flux_out_sanity_check = get_TSMstye_line(sz_dict, h_serp, u_res, resscale, interps, verbosity=1)
    global_water_losses.append(water_loss)
    global_water_losses_and_depths.append(water_losses_and_depths)
    layer_water_losses.append(layer_losses)
    global_flux_in.append(flux_in)
    global_flux_out.append(flux_out)
    global_flux_out_sanity_check.append(flux_out_sanity_check)


In [ ]:
fig, ax = pl.subplots()
s = list(allsz_params.keys())

for i in range(len(global_water_losses)):

    losses_at_depths = {'Depth (km)': [row[1] for row in(global_water_losses_and_depths[i])],
          'Water Loss (Tg/MYr/m)': global_water_losses[i]}

    df = pd.DataFrame(losses_at_depths)
    df.to_csv(output_folder / "{}_h2o_losses_at_depths_v2_{:.2f}_resscale_{:.2f}h_serp.csv".format(allsz_params[s[i]]['dirname'], resscale, h_serp))

    ax.plot(global_water_losses[i], [row[1] for row in(global_water_losses_and_depths[i])], label = [s[i]])


ax.yaxis.set_inverted(True)
ax.set_title("TSM")
ax.set_xlabel('slab H20 loss (Tg/MYr/m)')
ax.set_ylabel('depth (km)')
ax.set_box_aspect(0.67)
ax.set_ylim(250,50)
ax.set_xlim(0,35)
# fig.legend(loc='outside upper right')
pl.show()
fig.savefig(output_folder / ("TSM_v2_corrected_dist_increment"))

gwr = np.sum(global_flux_out)
print("Global Water Retention ---------- " , gwr)